In [ ]:
Calculo de densidades del grafo original UNAM2000-2024 y de entrenamiento, validacion y pruebas

In [10]:
import networkx as nx

# Rutas de los archivos GML
gml_files = [
    r"C:\Users\pilarang\0-ProyAcademicosGrafo-Integrado\2-ServSoc_Implementacion por fechas\BD_UNAM_Graph_2000_2020.gml",
    r"C:\Users\pilarang\0-ProyAcademicosGrafo-Integrado\2-ServSoc_Implementacion por fechas\BD_UNAM_Graph_2021_2022.gml",
    r"C:\Users\pilarang\0-ProyAcademicosGrafo-Integrado\2-ServSoc_Implementacion por fechas\BD_UNAM_Graph_2023_2024.gml"
]

# Lista para almacenar grafos individuales
graphs = []
for f in gml_files:
    G = nx.read_gml(f)
    graphs.append(G)
    dens = nx.density(G)
    print(f"Grafo '{f}' - Nodos: {G.number_of_nodes()}, Aristas: {G.number_of_edges()}, Densidad: {dens:.6f}")

# Crear grafo total combinado
G_total = nx.Graph()
for G in graphs:
    G_total.add_nodes_from(G.nodes(data=True))
    G_total.add_edges_from(G.edges(data=True))

# Densidad del grafo total
dens_total = nx.density(G_total)
print(f"\nGrafo total combinado - Nodos: {G_total.number_of_nodes()}, Aristas: {G_total.number_of_edges()}, Densidad: {dens_total:.6f}")

# Guardar grafo total
G_total_file = r"C:\Users\pilarang\0-ProyAcademicosGrafo-Integrado\2-ServSoc_Implementacion por fechas\BD_UNAM_Graph_total.gml"
nx.write_gml(G_total, G_total_file)
print(f"Grafo total guardado en: {G_total_file}")


Grafo 'C:\Users\pilarang\0-ProyAcademicosGrafo-Integrado\2-ServSoc_Implementacion por fechas\BD_UNAM_Graph_2000_2020.gml' - Nodos: 9041, Aristas: 14064, Densidad: 0.000344
Grafo 'C:\Users\pilarang\0-ProyAcademicosGrafo-Integrado\2-ServSoc_Implementacion por fechas\BD_UNAM_Graph_2021_2022.gml' - Nodos: 5151, Aristas: 6752, Densidad: 0.000509
Grafo 'C:\Users\pilarang\0-ProyAcademicosGrafo-Integrado\2-ServSoc_Implementacion por fechas\BD_UNAM_Graph_2023_2024.gml' - Nodos: 3207, Aristas: 4025, Densidad: 0.000783

Grafo total combinado - Nodos: 15750, Aristas: 24584, Densidad: 0.000198
Grafo total guardado en: C:\Users\pilarang\0-ProyAcademicosGrafo-Integrado\2-ServSoc_Implementacion por fechas\BD_UNAM_Graph_total.gml


In [ ]:
Recomendación práctica

Calcula la densidad del grafo original → como una referencia global.

Calcula la densidad de cada subgrafo (train, val, test) → para relacionar densidad local/temporal con rendimiento de modelos.

Analiza diferencias:

Si los modelos basados en LSP/GSP mejoran cuando sube la densidad, eso te muestra que la estructura topológica pesa más en redes densas.

Si FP se mantiene estable mientras LSP/GSP fluctúan con la densidad → conclusión: FP es más robusto a cambios estructurales.

👉 En resumen: debes calcular densidad en ambos niveles (grafo global y subgrafos), porque cada uno responde a una pregunta distinta:

Global → cómo es la red en general.

Subgrafos → cómo cambia el poder predictivo de FP, LSP, GSP según la evolución temporal.

Conversión de archivos de entrenamiento validacion y pruebas por fechas a FP,LSP y GSP separados

In [6]:
import pandas as pd

# Lista de archivos de entrada
file_paths = [
    r"C:\Users\pilarang\0-ProyAcademicosGrafo-Integrado\2-ServSoc_Implementacion por fechas\sample_train_2000_2020.csv",
    r"C:\Users\pilarang\0-ProyAcademicosGrafo-Integrado\2-ServSoc_Implementacion por fechas\sample_val_2021_2022.csv",
    r"C:\Users\pilarang\0-ProyAcademicosGrafo-Integrado\2-ServSoc_Implementacion por fechas\sample_test_2023_2024.csv"
]

# Columnas base que se mantienen en todos los archivos
base_cols = ["source", "target", "connected"]

# Diccionario de proximidad según la tabla
proximity_dict = {
    # Feature Proximity (FP)
    "sum_of_areas": "FP",
    "sum_of_papers": "FP",
    "sum_of_keywords": "FP",
    "keywords_match": "FP",

    # Local Structural Proximity (LSP)
    "lenght_short_path": "LSP",
    "common_neighbors": "LSP",
    "jaccard_coefficient": "LSP",
    "adamic_adar_index": "LSP",
    "resource_allocation": "LSP",
    "preferential_attachment": "LSP",
    "sum_of_neighbors": "LSP",
    "log_secundary_neighbors": "LSP",
    "clustering_index_sum": "LSP",
    "centralidad_grado_source": "LSP",
    "centralidad_grado_target": "LSP",
    "coeficiente_agrupamiento_source": "LSP",
    "coeficiente_agrupamiento_target": "LSP",

    # Global Structural Proximity (GSP)
    "community_cn": "GSP",
    "community_ra": "GSP",
    "closeness_centrality_source": "GSP",
    "closeness_centrality_target": "GSP",
    "betweenness_centrality_source": "GSP",
    "betweenness_centrality_target": "GSP",
    "eigenvector_centrality_source": "GSP",
    "eigenvector_centrality_target": "GSP",
    "pagerank_source": "GSP",
    "pagerank_target": "GSP",
    "k_core_number_source": "GSP",
    "k_core_number_target": "GSP"
}

# --- Procesar cada archivo ---
for file_path in file_paths:
    # Leer CSV
    df = pd.read_csv(file_path, index_col=0)  # Evita Unnamed:0 si existiera
    
    # Filtrar columnas existentes
    fp_cols_exist = [col for col, prox in proximity_dict.items() if prox=="FP" and col in df.columns]
    lsp_cols_exist = [col for col, prox in proximity_dict.items() if prox=="LSP" and col in df.columns]
    gsp_cols_exist = [col for col, prox in proximity_dict.items() if prox=="GSP" and col in df.columns]

    # Crear DataFrames filtrados
    df_fp = df[base_cols + fp_cols_exist] if fp_cols_exist else pd.DataFrame()
    df_lsp = df[base_cols + lsp_cols_exist] if lsp_cols_exist else pd.DataFrame()
    df_gsp = df[base_cols + gsp_cols_exist] if gsp_cols_exist else pd.DataFrame()

    # Guardar archivos
    fp_file = file_path.replace(".csv","_FP.csv")
    lsp_file = file_path.replace(".csv","_LSP.csv")
    gsp_file = file_path.replace(".csv","_GSP.csv")

    df_fp.to_csv(fp_file, index=False)
    df_lsp.to_csv(lsp_file, index=False)
    df_gsp.to_csv(gsp_file, index=False)

    # --- Resumen por archivo ---
    print(f"✅ Archivos generados para {file_path}:")
    print(f" - FP: {len(fp_cols_exist)} columnas ({fp_cols_exist}) -> {fp_file}")
    print(f" - LSP: {len(lsp_cols_exist)} columnas ({lsp_cols_exist}) -> {lsp_file}")
    print(f" - GSP: {len(gsp_cols_exist)} columnas ({gsp_cols_exist}) -> {gsp_file}")

    # Columnas no clasificadas
    all_classified_cols = set(fp_cols_exist + lsp_cols_exist + gsp_cols_exist + base_cols)
    unclassified_cols = [col for col in df.columns if col not in all_classified_cols]
    if unclassified_cols:
        print(f"⚠️ Columnas no clasificadas: {unclassified_cols}")
    print("--------------------------------------------------")


✅ Archivos generados para C:\Users\pilarang\0-ProyAcademicosGrafo-Integrado\2-ServSoc_Implementacion por fechas\sample_train_2000_2020.csv:
 - FP: 4 columnas (['sum_of_areas', 'sum_of_papers', 'sum_of_keywords', 'keywords_match']) -> C:\Users\pilarang\0-ProyAcademicosGrafo-Integrado\2-ServSoc_Implementacion por fechas\sample_train_2000_2020_FP.csv
 - LSP: 8 columnas (['lenght_short_path', 'jaccard_coefficient', 'adamic_adar_index', 'resource_allocation', 'preferential_attachment', 'sum_of_neighbors', 'log_secundary_neighbors', 'clustering_index_sum']) -> C:\Users\pilarang\0-ProyAcademicosGrafo-Integrado\2-ServSoc_Implementacion por fechas\sample_train_2000_2020_LSP.csv
 - GSP: 2 columnas (['community_cn', 'community_ra']) -> C:\Users\pilarang\0-ProyAcademicosGrafo-Integrado\2-ServSoc_Implementacion por fechas\sample_train_2000_2020_GSP.csv
--------------------------------------------------
✅ Archivos generados para C:\Users\pilarang\0-ProyAcademicosGrafo-Integrado\2-ServSoc_Implementac

In [5]:
# ESTE PROGRAMA DIVIDE LOS ARCHIVOS POR FP, LSP, GSP Y SUS COMBINACIONES A LOS ARCHIVOS DE TRAIN, VAL Y TEST DE LA UNAM
import pandas as pd

# Lista de archivos de entrada
file_paths = [
    r"C:\Users\pilarang\0-ProyAcademicosGrafo-Integrado\2-ServSoc_Implementacion por fechas\sample_train_2000_2020.csv",
    r"C:\Users\pilarang\0-ProyAcademicosGrafo-Integrado\2-ServSoc_Implementacion por fechas\sample_val_2021_2022.csv",
    r"C:\Users\pilarang\0-ProyAcademicosGrafo-Integrado\2-ServSoc_Implementacion por fechas\sample_test_2023_2024.csv"
]

# Columnas base que se mantienen en todos los archivos
base_cols = ["source", "target", "connected"]

# Diccionario de proximidad según la tabla
proximity_dict = {
    # Feature Proximity (FP)
    "sum_of_areas": "FP",
    "sum_of_papers": "FP",
    "sum_of_keywords": "FP",
    "keywords_match": "FP",

    # Local Structural Proximity (LSP)
    "lenght_short_path": "LSP",
    "common_neighbors": "LSP",
    "jaccard_coefficient": "LSP",
    "adamic_adar_index": "LSP",
    "resource_allocation": "LSP",
    "preferential_attachment": "LSP",
    "sum_of_neighbors": "LSP",
    "log_secundary_neighbors": "LSP",
    "clustering_index_sum": "LSP",
    "centralidad_grado_source": "LSP",
    "centralidad_grado_target": "LSP",
    "coeficiente_agrupamiento_source": "LSP",
    "coeficiente_agrupamiento_target": "LSP",

    # Global Structural Proximity (GSP)
    "community_cn": "GSP",
    "community_ra": "GSP",
    "closeness_centrality_source": "GSP",
    "closeness_centrality_target": "GSP",
    "betweenness_centrality_source": "GSP",
    "betweenness_centrality_target": "GSP",
    "eigenvector_centrality_source": "GSP",
    "eigenvector_centrality_target": "GSP",
    "pagerank_source": "GSP",
    "pagerank_target": "GSP",
    "k_core_number_source": "GSP",
    "k_core_number_target": "GSP"
}

# --- Procesar cada archivo ---
for file_path in file_paths:
    # Leer CSV
    df = pd.read_csv(file_path, index_col=0)  # Evita Unnamed:0 si existiera
    
    # Filtrar columnas existentes
    fp_cols_exist = [col for col, prox in proximity_dict.items() if prox=="FP" and col in df.columns]
    lsp_cols_exist = [col for col, prox in proximity_dict.items() if prox=="LSP" and col in df.columns]
    gsp_cols_exist = [col for col, prox in proximity_dict.items() if prox=="GSP" and col in df.columns]

    # Crear DataFrames filtrados
    df_fp = df[base_cols + fp_cols_exist] if fp_cols_exist else pd.DataFrame()
    df_lsp = df[base_cols + lsp_cols_exist] if lsp_cols_exist else pd.DataFrame()
    df_gsp = df[base_cols + gsp_cols_exist] if gsp_cols_exist else pd.DataFrame()

    # Guardar archivos individuales
    #fp_file = file_path.replace(".csv","_FP.csv")
    #lsp_file = file_path.replace(".csv","_LSP.csv")
    gsp_file = file_path.replace(".csv","_GSP.csv")

    #df_fp.to_csv(fp_file, index=False)
    #df_lsp.to_csv(lsp_file, index=False)
    df_gsp.to_csv(gsp_file, index=False)

    # --- Archivos combinados ---
    #lsp_gsp_file = file_path.replace(".csv","_LSP_GSP.csv")
    #gsp_fp_file  = file_path.replace(".csv","_GSP_FP.csv")
    #lsp_fp_file  = file_path.replace(".csv","_LSP_FP.csv")

    #df_lsp_gsp = df[base_cols + lsp_cols_exist + gsp_cols_exist] if (lsp_cols_exist or gsp_cols_exist) else pd.DataFrame()
    #df_gsp_fp  = df[base_cols + gsp_cols_exist + fp_cols_exist] if (gsp_cols_exist or fp_cols_exist) else pd.DataFrame()
    #df_lsp_fp  = df[base_cols + lsp_cols_exist + fp_cols_exist] if (lsp_cols_exist or fp_cols_exist) else pd.DataFrame()

    #df_lsp_gsp.to_csv(lsp_gsp_file, index=False)
    #df_gsp_fp.to_csv(gsp_fp_file, index=False)
    #df_lsp_fp.to_csv(lsp_fp_file, index=False)

    # --- Resumen por archivo ---
    print(f"✅ Archivos generados para {file_path}:")
    #print(f" - FP: {len(fp_cols_exist)} columnas -> {fp_file}")
    #print(f" - LSP: {len(lsp_cols_exist)} columnas -> {lsp_file}")
    print(f" - GSP: {len(gsp_cols_exist)} columnas -> {gsp_file}")
    #print(f" - LSP+GSP: {len(lsp_cols_exist)+len(gsp_cols_exist)} columnas -> {lsp_gsp_file}")
    #print(f" - GSP+FP: {len(gsp_cols_exist)+len(fp_cols_exist)} columnas -> {gsp_fp_file}")
    #print(f" - LSP+FP: {len(lsp_cols_exist)+len(fp_cols_exist)} columnas -> {lsp_fp_file}")

    # Columnas no clasificadas
    all_classified_cols = set(fp_cols_exist + lsp_cols_exist + gsp_cols_exist + base_cols)
    unclassified_cols = [col for col in df.columns if col not in all_classified_cols]
    if unclassified_cols:
        print(f"⚠️ Columnas no clasificadas: {unclassified_cols}")
    print("--------------------------------------------------")


✅ Archivos generados para C:\Users\pilarang\0-ProyAcademicosGrafo-Integrado\2-ServSoc_Implementacion por fechas\sample_train_2000_2020.csv:
 - GSP: 2 columnas -> C:\Users\pilarang\0-ProyAcademicosGrafo-Integrado\2-ServSoc_Implementacion por fechas\sample_train_2000_2020_GSP.csv
--------------------------------------------------
✅ Archivos generados para C:\Users\pilarang\0-ProyAcademicosGrafo-Integrado\2-ServSoc_Implementacion por fechas\sample_val_2021_2022.csv:
 - GSP: 2 columnas -> C:\Users\pilarang\0-ProyAcademicosGrafo-Integrado\2-ServSoc_Implementacion por fechas\sample_val_2021_2022_GSP.csv
--------------------------------------------------
✅ Archivos generados para C:\Users\pilarang\0-ProyAcademicosGrafo-Integrado\2-ServSoc_Implementacion por fechas\sample_test_2023_2024.csv:
 - GSP: 2 columnas -> C:\Users\pilarang\0-ProyAcademicosGrafo-Integrado\2-ServSoc_Implementacion por fechas\sample_test_2023_2024_GSP.csv
--------------------------------------------------
